In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

In [2]:
import ROOT
from analysis_framework import Dataset
from OptimalObservableHelper import OptimalObservableHelper
from AltSetupHandler import AltSetupHandler

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x95e51b0
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x96a6c10


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 12
# n_threads = 6
no_rvec = True
# write_outputs = False
write_outputs = True
dataset_path = "data/datasets/reweighted/signal-only.json"
# friend_dataset_path = "data/datasets/truth-reweighted/signal-only.json"
# friend_dataset_path = "data/datasets/truth-reweighted-hel/signal-only.json"
friend_dataset_path = "data/datasets/truth-reweighted-hel-mW/signal-only.json"

plot_path = "plots/oo-val"

out_dir = "fit-configs/signal-only-mW-pol"

do_scaling_plots = False

e_pol = 0. # 0.8
p_pol = 0. # 0.6


In [4]:
ROOT.EnableImplicitMT(n_threads)
# environ["OMP_NUM_THREADS"] = "6"

In [5]:
dataset = Dataset.from_json(dataset_path)
friend_dataset = Dataset.from_json(friend_dataset_path)

In [6]:
analysis = OptimalObservableHelper(dataset, friend_datasets=[friend_dataset])

missing friend for sample: 4f_sw_sl_eLpL_bkg
missing friend for sample: 4f_sw_sl_eLpR_bkg
missing friend for sample: 4f_sw_sl_eRpR_bkg
missing friend for sample: 4f_sw_sl_eRpL_bkg
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xd41bb50


In [7]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]

True


In [8]:
slope_comp_g_orders = list(range(3, 17))

oo_calc_order = 8
# oo_calc_order = 5

oo_configs = [
    f"g1z_pos_1em{oo_calc_order:02}",
    f"ka_pos_1em{oo_calc_order:02}",
    f"la_pos_1em{oo_calc_order:02}",
    f"mW_pos_1em{oo_calc_order:02}",
    ]

if do_scaling_plots:
    oo_config_g_val = [f"g1z_pos_1em{x:02}" for x in slope_comp_g_orders]
    oo_config_g_val += [f"ka_pos_1em{x:02}" for x in slope_comp_g_orders]
    oo_config_g_val += [f"la_pos_1em{x:02}" for x in slope_comp_g_orders]
    oo_config_g_val += [f"mW_pos_1em{x:02}" for x in slope_comp_g_orders]
else:
    oo_config_g_val = oo_configs

# oo_names = [f"O_{c}" for c in oo_configs]
oo_names = {
    # "mlvec_reco_oo": analysis.define_optimal_observables("mlvec_O", ["mlvec_reco_sqme", "wj_mlvec_reco_sqme"], oo_configs, categories=signal_category),
    # "reco_oo": analysis.define_optimal_observables("O", ["reco_sqme", "wj_reco_sqme"], oo_configs, categories=signal_category),
    # "reco_jm_oo": analysis.define_optimal_observables("jm_O", ["reco_sqme"], oo_configs, categories=signal_category),
    # "clean_reco_oo": analysis.define_optimal_observables("clean_O", ["clean_reco_sqme", "wj_clean_reco_sqme"], oo_configs, categories=signal_category),
    # "clean_reco_jm_oo": analysis.define_optimal_observables("clean_jm_O", ["clean_reco_sqme"], oo_configs, categories=signal_category),
    # "cheat_clean_reco_oo": analysis.define_optimal_observables("cheat_clean_O", ["cheat_clean_reco_sqme", "wj_cheat_clean_reco_sqme"], oo_configs, categories=signal_category),
    # "cheat_clean_reco_jm_oo": analysis.define_optimal_observables("cheat_clean_jm_O", ["cheat_clean_reco_sqme"], oo_configs, categories=signal_category),
    # "mc_oo": analysis.define_optimal_observables("mc_O", ["mc_sqme"], oo_config_g_val, categories=signal_category),
    # "mc_oo": analysis.define_optimal_observables_polarised("mc_O", ["mc_sqme_hels"], oo_config_g_val, categories=signal_category),
    "mc_oo": analysis.define_optimal_observables_polarised("mc_O", ["mc_sqme_hels"], oo_config_g_val, categories=signal_category, nom_epol=e_pol, nom_ppol=p_pol, add_pol_oo=True),
    # "nomb_mc_oo": analysis.define_optimal_observables("nomb_mc_O", ["nomb_mc_sqme"], oo_configs, categories=signal_category),
    # "nomb_mc_rlep_oo": analysis.define_optimal_observables("nomb_mc_rlep_O", ["nomb_mc_rlep_sqme"], oo_configs, categories=signal_category),
    # "av_mc_oo": analysis.define_optimal_observables("av_mc_O", ["mc_sqme", "wj_mc_sqme"], oo_configs, categories=signal_category),
    # "av_nomb_mc_oo": analysis.define_optimal_observables("av_nomb_mc_O", ["nomb_mc_sqme", "wj_nomb_mc_sqme"], oo_configs, categories=signal_category),
    # "av_nomb_mc_rlep_oo": analysis.define_optimal_observables("av_nomb_mc_rlep_O", ["nomb_mc_rlep_sqme", "wj_nomb_mc_rlep_sqme"], oo_configs, categories=signal_category),
}

In [9]:
for names in oo_names.values():
    for name in names:
        # create dummy oo where they don't exist yet
        analysis.define_only_on(["4f_sl_bkg"], name, "0.")
# TODO: re-consider what is correct here...
analysis.add_filter("&&".join([f"std::isfinite({oo})" for oo_name in oo_names.values() for oo in oo_name]), "finite OO")
# analysis.add_filter("&&".join([f"abs({oo}) <= 5" for oo in oo_names["reco_oo"]]), "abs(OO) <= 5")
# analysis.add_filter("(iso_lep_lvec + nu_lvec).M() > 0.", "physical nu")
# analysis.add_filter("(iso_lep_lvec + clean_nu_lvec).M() > 0.", "physical clean_nu")
# analysis.add_filter("nu_lvec.E() > 0.", "physical nu")
# analysis.add_filter("clean_nu_lvec.E() > 0.", "physical clean nu")
analysis.book_reports()

In [10]:
for on in oo_names["mc_oo"]:
    analysis.book_histogram_1D(on, on, ("", "", 250, -2.5, 2.5), categories=signal_category)

In [11]:
alt_setup_handler = AltSetupHandler("""
{
  "SM": {
    "mW": 80.419,
    "g1z": 1.0,
    "ka": 1.0,
    "la": 0.0
  },
"variations": [
    0.1,
    -0.1,
    0.01,
    -0.01,
    0.002,
    -0.002,
    0.001,
    -0.001,
    7.5e-04,
    -7.5e-04,
    5e-04,
    -5e-04,
    2.5e-04,
    -2.5e-04,
    1e-04,
    -1e-04,
    1e-05,
    1e-06,
    1e-07,
    1e-08,
    1e-09,
    1e-10,
    1e-11,
    1e-12,
    1e-13,
    1e-14,
    1e-15,
    1e-16
  ]
}
""", mirror=False, combinations=False)
alt_config_names = list(alt_setup_handler.get_alt_setup().keys())

In [12]:
weight_names = analysis.book_weight_sums(["nominal"] + alt_config_names, categories=signal_category, hel=True)
# weight_names = analysis.book_weight_sums(["nominal"] + [name for name in alt_config_names if "em03" in name] + [name for name in alt_config_names if "em04" in name], categories=signal_category)

In [13]:
print(weight_names)

['weight_hel_nominal', 'weight_hel_mW_pos_1em01', 'weight_hel_g1z_pos_1em01', 'weight_hel_ka_pos_1em01', 'weight_hel_la_pos_1em01', 'weight_hel_mW_neg_1em01', 'weight_hel_g1z_neg_1em01', 'weight_hel_ka_neg_1em01', 'weight_hel_la_neg_1em01', 'weight_hel_mW_pos_1em02', 'weight_hel_g1z_pos_1em02', 'weight_hel_ka_pos_1em02', 'weight_hel_la_pos_1em02', 'weight_hel_mW_neg_1em02', 'weight_hel_g1z_neg_1em02', 'weight_hel_ka_neg_1em02', 'weight_hel_la_neg_1em02', 'weight_hel_mW_pos_2em03', 'weight_hel_g1z_pos_2em03', 'weight_hel_ka_pos_2em03', 'weight_hel_la_pos_2em03', 'weight_hel_mW_neg_2em03', 'weight_hel_g1z_neg_2em03', 'weight_hel_ka_neg_2em03', 'weight_hel_la_neg_2em03', 'weight_hel_mW_pos_1em03', 'weight_hel_g1z_pos_1em03', 'weight_hel_ka_pos_1em03', 'weight_hel_la_pos_1em03', 'weight_hel_mW_neg_1em03', 'weight_hel_g1z_neg_1em03', 'weight_hel_ka_neg_1em03', 'weight_hel_la_neg_1em03', 'weight_hel_mW_pos_8em04', 'weight_hel_g1z_pos_8em04', 'weight_hel_ka_pos_8em04', 'weight_hel_la_pos_8em0

In [14]:
for names in oo_names.values():
    analysis.define_weighted_oo(names, weight_names, categories=signal_category)
    analysis.book_oo_sums(names, weight_names, categories=signal_category)
    if not do_scaling_plots:
        analysis.book_oo_matrix(names, categories=signal_category)

In [15]:
pars = ["g1z", "ka", "la", "mW", "epol", "ppol"]
if do_scaling_plots:
    for x in slope_comp_g_orders:
        analysis.book_oo_matrix([f"mc_O_{p}_pos_1em{x:02}" for p in pars])

In [16]:
%%time
analysis.run()

CPU times: user 2min 44s, sys: 6.22 s, total: 2min 51s
Wall time: 33.6 s


In [17]:
analysis.print_reports()

         4f_sw_sl_signal               4f_sl_bkg
        10244400 (1e-03)           33239 (2e-02) All
        10244400 (1e-03)           33239 (2e-02) finite OO
                    1.00                    1.00 efficiency



In [18]:
if write_outputs:
    for name, names in oo_names.items():
        print(name)
        analysis.print_fit_input(names, weight_names, pars, e_pol=e_pol, p_pol=p_pol, categories=signal_category, dir=out_dir, name=name, hel=True)

mc_oo
# pars, evt/ab_inv, means, slopes, cov
['g1z', 'ka', 'la', 'mW', 'epol', 'ppol']
2048879.9724151532
[-0.10403988082763777, -0.06933219941656077, 0.09042444753289121, 0.09601638231170931, -0.9612242349304312, 0.9610860792217243]
[[-14.252107411432382, 0.23519678607448383, 8.170251336328695, -0.10889928840087823, -0.5467272149146835, 0.5489337910392017], [0.40729348556810585, -6.377662661142151, 0.9386536609817574, -0.18074137296839607, 0.41204386997708925, -0.4062403043774893], [-9.290141128260881, -0.8084869341175713, 26.579183636790745, 0.02428000763822517, -0.030024702545253867, 0.0314175697892897], [0.11522729784733207, 0.13075765073137124, 0.02776551032622851, 9.565104810786352, -0.05572847076948713, 0.05620007605962516], [-0.059943266775737414, 0.029626441504230223, 0.003926431347234898, 0.005538951517433903, -0.03203506345187611, 0.009589661875859476], [-0.05984273106808107, 0.029339064703394608, 0.004127095257992203, 0.005557283302345887, -0.009593212115419473, 0.032275532

In [19]:
print(alt_config_names)

['mW_pos_1em01', 'g1z_pos_1em01', 'ka_pos_1em01', 'la_pos_1em01', 'mW_neg_1em01', 'g1z_neg_1em01', 'ka_neg_1em01', 'la_neg_1em01', 'mW_pos_1em02', 'g1z_pos_1em02', 'ka_pos_1em02', 'la_pos_1em02', 'mW_neg_1em02', 'g1z_neg_1em02', 'ka_neg_1em02', 'la_neg_1em02', 'mW_pos_2em03', 'g1z_pos_2em03', 'ka_pos_2em03', 'la_pos_2em03', 'mW_neg_2em03', 'g1z_neg_2em03', 'ka_neg_2em03', 'la_neg_2em03', 'mW_pos_1em03', 'g1z_pos_1em03', 'ka_pos_1em03', 'la_pos_1em03', 'mW_neg_1em03', 'g1z_neg_1em03', 'ka_neg_1em03', 'la_neg_1em03', 'mW_pos_8em04', 'g1z_pos_8em04', 'ka_pos_8em04', 'la_pos_8em04', 'mW_neg_8em04', 'g1z_neg_8em04', 'ka_neg_8em04', 'la_neg_8em04', 'mW_pos_5em04', 'g1z_pos_5em04', 'ka_pos_5em04', 'la_pos_5em04', 'mW_neg_5em04', 'g1z_neg_5em04', 'ka_neg_5em04', 'la_neg_5em04', 'mW_pos_3em04', 'g1z_pos_3em04', 'ka_pos_3em04', 'la_pos_3em04', 'mW_neg_3em04', 'g1z_neg_3em04', 'ka_neg_3em04', 'la_neg_3em04', 'mW_pos_1em04', 'g1z_pos_1em04', 'ka_pos_1em04', 'la_pos_1em04', 'mW_neg_1em04', 'g1z_neg

In [20]:
# calculate means etc.
names = oo_names["mc_oo"]
# names = oo_names["mlvec_reco_oo"]

oo_means = analysis.calc_oo_means(names, weight_names, categories=signal_category, vary_pol=True)
# oo_means = analysis.calc_oo_means(names, weight_names, categories=signal_category)


In [21]:
print(names)

['mc_O_g1z_pos_1em08', 'mc_O_ka_pos_1em08', 'mc_O_la_pos_1em08', 'mc_O_mW_pos_1em08', 'mc_O_epol', 'mc_O_ppol']


In [22]:
print(oo_means.keys())

dict_keys(['mc_O_g1z_pos_1em08_weight_hel_nominal', 'mc_O_ka_pos_1em08_weight_hel_nominal', 'mc_O_la_pos_1em08_weight_hel_nominal', 'mc_O_mW_pos_1em08_weight_hel_nominal', 'mc_O_epol_weight_hel_nominal', 'mc_O_ppol_weight_hel_nominal', 'mc_O_g1z_pos_1em08_weight_hel_mW_pos_1em01', 'mc_O_ka_pos_1em08_weight_hel_mW_pos_1em01', 'mc_O_la_pos_1em08_weight_hel_mW_pos_1em01', 'mc_O_mW_pos_1em08_weight_hel_mW_pos_1em01', 'mc_O_epol_weight_hel_mW_pos_1em01', 'mc_O_ppol_weight_hel_mW_pos_1em01', 'mc_O_g1z_pos_1em08_weight_hel_g1z_pos_1em01', 'mc_O_ka_pos_1em08_weight_hel_g1z_pos_1em01', 'mc_O_la_pos_1em08_weight_hel_g1z_pos_1em01', 'mc_O_mW_pos_1em08_weight_hel_g1z_pos_1em01', 'mc_O_epol_weight_hel_g1z_pos_1em01', 'mc_O_ppol_weight_hel_g1z_pos_1em01', 'mc_O_g1z_pos_1em08_weight_hel_ka_pos_1em01', 'mc_O_ka_pos_1em08_weight_hel_ka_pos_1em01', 'mc_O_la_pos_1em08_weight_hel_ka_pos_1em01', 'mc_O_mW_pos_1em08_weight_hel_ka_pos_1em01', 'mc_O_epol_weight_hel_ka_pos_1em01', 'mc_O_ppol_weight_hel_ka_pos_1

In [23]:
if do_scaling_plots:
    oo_mat_g_val_graphs = []

    from itertools import combinations_with_replacement
    for i, (p1, p2) in enumerate(combinations_with_replacement(pars, 2)):
        oo_mat_g_val_graphs.append(ROOT.TGraph())
        # could set title already here as I have p1 and p2 available

    for x in slope_comp_g_orders:
        mat = analysis.get_oo_matrix([f"mc_O_{p}_pos_1em{x:02}" for p in pars])
        for i, (p1, p2) in enumerate(combinations_with_replacement(pars, 2)):
            oo_mat_g_val_graphs[i].AddPoint(x, mat[i])

    oo_mat_val_canvs = []
    for i, (p1, p2) in enumerate(combinations_with_replacement(pars, 2)):
        c = ROOT.TCanvas()
        oo_mat_g_val_graphs[i].Draw("alp")
        c.Draw()
        oo_mat_val_canvs.append(c)


In [24]:
if do_scaling_plots:
    mean_g_val_graphs = {}

    for par in pars:
        graph = ROOT.TGraph()
        for x in slope_comp_g_orders:
            # mean = oo_means[f"mc_O_{par}_pos_1em{x:02}_weight_nominal"]
            mean = oo_means[f"mc_O_{par}_pos_1em{x:02}_weight_hel_nominal"]
            graph.AddPoint(x, mean)

        mean_g_val_graphs[par] = graph

    mgval_max = {
        "g1z": -0.10255,
        "ka": -0.06895,
        "la": 0.092,
    }
    mgval_min = {
        "g1z": -0.1044,
        "ka": -0.06938,
        "la": 0.09025,
    }
    mgval_canvs = {}
    for k, graph in mean_g_val_graphs.items():
        c = ROOT.TCanvas()
        graph.SetTitle(f";-log_{{10}}(#Delta {k}); E_{{{0}}}[O_{{{k}}}]")
        if k in mgval_min:
            graph.SetMinimum(mgval_min[k])
        if k in mgval_max:
            graph.SetMaximum(mgval_max[k])
        graph.Draw("alp")
        c.Draw()
        c.SaveAs(f"{plot_path}/mean_stab_{k}.pdf")
        mgval_canvs[k] = c


In [25]:
# print(oo_means.keys())
test = []
for key in oo_means:
    if "mc_O_g1z" in key:
        test.append(key)

print(test)

['mc_O_g1z_pos_1em08_weight_hel_nominal', 'mc_O_g1z_pos_1em08_weight_hel_mW_pos_1em01', 'mc_O_g1z_pos_1em08_weight_hel_g1z_pos_1em01', 'mc_O_g1z_pos_1em08_weight_hel_ka_pos_1em01', 'mc_O_g1z_pos_1em08_weight_hel_la_pos_1em01', 'mc_O_g1z_pos_1em08_weight_hel_mW_neg_1em01', 'mc_O_g1z_pos_1em08_weight_hel_g1z_neg_1em01', 'mc_O_g1z_pos_1em08_weight_hel_ka_neg_1em01', 'mc_O_g1z_pos_1em08_weight_hel_la_neg_1em01', 'mc_O_g1z_pos_1em08_weight_hel_mW_pos_1em02', 'mc_O_g1z_pos_1em08_weight_hel_g1z_pos_1em02', 'mc_O_g1z_pos_1em08_weight_hel_ka_pos_1em02', 'mc_O_g1z_pos_1em08_weight_hel_la_pos_1em02', 'mc_O_g1z_pos_1em08_weight_hel_mW_neg_1em02', 'mc_O_g1z_pos_1em08_weight_hel_g1z_neg_1em02', 'mc_O_g1z_pos_1em08_weight_hel_ka_neg_1em02', 'mc_O_g1z_pos_1em08_weight_hel_la_neg_1em02', 'mc_O_g1z_pos_1em08_weight_hel_mW_pos_2em03', 'mc_O_g1z_pos_1em08_weight_hel_g1z_pos_2em03', 'mc_O_g1z_pos_1em08_weight_hel_ka_pos_2em03', 'mc_O_g1z_pos_1em08_weight_hel_la_pos_2em03', 'mc_O_g1z_pos_1em08_weight_hel_mW

In [26]:
oo_slopes = analysis.get_slopes(names, pars, oo_means, g=1e-4, hel=True)

In [27]:
print(oo_slopes.keys())
print(names)

dict_keys(['mc_O_g1z_pos_1em08_g1z', 'mc_O_g1z_pos_1em08_ka', 'mc_O_g1z_pos_1em08_la', 'mc_O_g1z_pos_1em08_mW', 'mc_O_g1z_pos_1em08_epol', 'mc_O_g1z_pos_1em08_ppol', 'mc_O_ka_pos_1em08_g1z', 'mc_O_ka_pos_1em08_ka', 'mc_O_ka_pos_1em08_la', 'mc_O_ka_pos_1em08_mW', 'mc_O_ka_pos_1em08_epol', 'mc_O_ka_pos_1em08_ppol', 'mc_O_la_pos_1em08_g1z', 'mc_O_la_pos_1em08_ka', 'mc_O_la_pos_1em08_la', 'mc_O_la_pos_1em08_mW', 'mc_O_la_pos_1em08_epol', 'mc_O_la_pos_1em08_ppol', 'mc_O_mW_pos_1em08_g1z', 'mc_O_mW_pos_1em08_ka', 'mc_O_mW_pos_1em08_la', 'mc_O_mW_pos_1em08_mW', 'mc_O_mW_pos_1em08_epol', 'mc_O_mW_pos_1em08_ppol', 'mc_O_epol_g1z', 'mc_O_epol_ka', 'mc_O_epol_la', 'mc_O_epol_mW', 'mc_O_epol_epol', 'mc_O_epol_ppol', 'mc_O_ppol_g1z', 'mc_O_ppol_ka', 'mc_O_ppol_la', 'mc_O_ppol_mW', 'mc_O_ppol_epol', 'mc_O_ppol_ppol'])
['mc_O_g1z_pos_1em08', 'mc_O_ka_pos_1em08', 'mc_O_la_pos_1em08', 'mc_O_mW_pos_1em08', 'mc_O_epol', 'mc_O_ppol']


In [28]:
# oo_suffix = "_pos_1em08"

# slope_val_graphs = {k.replace(oo_suffix, ""): ROOT.TGraph() for k in oo_slopes}

# for x in slope_comp_g_orders:
#     g = 10**(-x)
#     # print(g)
#     slopes = analysis.get_slopes(names, pars, oo_means, g=g)
#     for k, s in slopes.items():
#         key = k.replace(oo_suffix, "")
#         slope_val_graphs[key].AddPoint(x, s)

# slope_val_canvs = {}
# for k, g in slope_val_graphs.items():
#     c = ROOT.TCanvas()
#     g.SetTitle(k)
#     g.Draw("alp")
#     c.Draw()
#     slope_val_canvs[k] = c

In [29]:
# x_points = [-2e-3, -1.5e-3, -1e-3, -5e-4, 5e-4, 1e-3, 1.5e-3, 2e-3]
# x_points = [-1.5e-3, -1e-3, -5e-4, -2.5e-4, -1e-4, 1e-4, 2.5e-4, 5e-4, 1e-3, 1.5e-3]
x_points = [-2e-3, -1e-3, -5e-4, -2.5e-4, -1e-4, 1e-4, 2.5e-4, 5e-4, 1e-3, 2e-3]
# oo_graphs = analysis.make_slope_graphs([f"mc_O_{p}_pos_1em{oo_calc_order:02}" for p in pars], pars, x_points, oo_means, hel=True)
oo_graphs = analysis.make_slope_graphs(oo_names["mc_oo"], pars, x_points, oo_means, hel=True)

In [30]:
canvases = {}
f_slopes = {}
r_graphs = {}
for k, g in oo_graphs.items():
    print(k)
    x_unit = k.split("_")[-1]
    o_unit = k.removesuffix(f"_{x_unit}").removesuffix(f"_pos_1em{oo_calc_order:02}").split("_")[-1]
    c = ROOT.TCanvas()
    left_margin = 0.23
    # left_margin = ROOT.gStyle.GetPadLeftMargin()
    right_margin = 0.05
    bottom_ratio = 0.5
    top_ratio = 1. - bottom_ratio
    c.DivideRatios(1, 2, [1.], [top_ratio, bottom_ratio])
    c.cd(1)
    c.GetPad(1).SetBottomMargin(0.)
    c.GetPad(1).SetLeftMargin(left_margin)
    c.GetPad(1).SetRightMargin(right_margin)
    g.SetTitle(f";;#frac{{#Delta E[O_{{{o_unit}}}] }}{{E_{{0}}[O_{{{o_unit}}}]}}[%]")
    g.Draw("alp")
    g.GetYaxis().SetLabelSize(ROOT.gStyle.GetLabelSize() * (1 / top_ratio))
    g.GetYaxis().SetTitleSize(ROOT.gStyle.GetTitleSize() * 0.85 * (1 / top_ratio))
    g.GetYaxis().SetTitleOffset(ROOT.gStyle.GetTitleOffset()* 1.6 * top_ratio)
    # f = ROOT.TF1(f"f_{k}", f"{oo_slopes[k]} * x + 1.", -0.01, 0.01)
    # f = ROOT.TF1(f"f_{k}", f"{oo_slopes[k]} * x", -0.01, 0.01)
    f = ROOT.TF1(f"f_{k}", f"{oo_slopes[k]} * x * 100", -0.01, 0.01)
    f.Draw("same")
    # f.Draw()
    f_slopes[k] = f
    c.cd(2)
    c.GetPad(2).SetTopMargin(0.)
    c.GetPad(2).SetBottomMargin(0.175 * (1. / bottom_ratio))
    c.GetPad(2).SetLeftMargin(left_margin)
    c.GetPad(2).SetRightMargin(right_margin)
    r_graph = analysis.make_ratio_graph(g, f)
    r_graph.Draw("alp")
    r_graph.GetYaxis().SetLabelSize(ROOT.gStyle.GetLabelSize() * (1 / bottom_ratio))
    r_graph.GetXaxis().SetLabelSize(ROOT.gStyle.GetLabelSize() * (1 / bottom_ratio))
    r_graph.GetXaxis().SetTitleSize(ROOT.gStyle.GetTitleSize() * (1 / bottom_ratio))
    r_graph.GetYaxis().SetTitleSize(ROOT.gStyle.GetTitleSize() * 0.7 * (1 / top_ratio))
    r_graph.GetYaxis().SetTitleOffset(ROOT.gStyle.GetTitleOffset()* 1.6 * top_ratio)
    r_graph.SetTitle(f";{x_unit};#splitline{{difference}}{{to linear}} [10^{{-6}}]")
    r_graphs[k] = r_graph
    c.Draw()
    c.SaveAs(f"{plot_path}/ratio_{k}.pdf")
    canvases[k] = c

mc_O_g1z_pos_1em08_g1z
mc_O_g1z_pos_1em08_ka
mc_O_g1z_pos_1em08_la
mc_O_g1z_pos_1em08_mW
mc_O_g1z_pos_1em08_epol
mc_O_g1z_pos_1em08_ppol
mc_O_ka_pos_1em08_g1z
mc_O_ka_pos_1em08_ka
mc_O_ka_pos_1em08_la
mc_O_ka_pos_1em08_mW
mc_O_ka_pos_1em08_epol
mc_O_ka_pos_1em08_ppol
mc_O_la_pos_1em08_g1z
mc_O_la_pos_1em08_ka
mc_O_la_pos_1em08_la
mc_O_la_pos_1em08_mW
mc_O_la_pos_1em08_epol
mc_O_la_pos_1em08_ppol
mc_O_mW_pos_1em08_g1z
mc_O_mW_pos_1em08_ka
mc_O_mW_pos_1em08_la
mc_O_mW_pos_1em08_mW
mc_O_mW_pos_1em08_epol
mc_O_mW_pos_1em08_ppol
mc_O_epol_g1z
mc_O_epol_ka
mc_O_epol_la
mc_O_epol_mW
mc_O_epol_epol
mc_O_epol_ppol
mc_O_ppol_g1z
mc_O_ppol_ka
mc_O_ppol_la
mc_O_ppol_mW
mc_O_ppol_epol
mc_O_ppol_ppol


Info in <TCanvas::Print>: pdf file plots/oo-val/ratio_mc_O_g1z_pos_1em08_g1z.pdf has been created
Info in <TCanvas::Print>: pdf file plots/oo-val/ratio_mc_O_g1z_pos_1em08_ka.pdf has been created
Info in <TCanvas::Print>: pdf file plots/oo-val/ratio_mc_O_g1z_pos_1em08_la.pdf has been created
Info in <TCanvas::Print>: pdf file plots/oo-val/ratio_mc_O_g1z_pos_1em08_mW.pdf has been created
Info in <TCanvas::Print>: pdf file plots/oo-val/ratio_mc_O_g1z_pos_1em08_epol.pdf has been created
Info in <TCanvas::Print>: pdf file plots/oo-val/ratio_mc_O_g1z_pos_1em08_ppol.pdf has been created
Info in <TCanvas::Print>: pdf file plots/oo-val/ratio_mc_O_ka_pos_1em08_g1z.pdf has been created
Info in <TCanvas::Print>: pdf file plots/oo-val/ratio_mc_O_ka_pos_1em08_ka.pdf has been created
Info in <TCanvas::Print>: pdf file plots/oo-val/ratio_mc_O_ka_pos_1em08_la.pdf has been created
Info in <TCanvas::Print>: pdf file plots/oo-val/ratio_mc_O_ka_pos_1em08_mW.pdf has been created
Info in <TCanvas::Print>: pd

In [31]:
for on in oo_names["mc_oo"]:
    analysis.draw_histogram(on, categories=signal_category, e_pol=e_pol, p_pol=p_pol)

In [32]:
for on in oo_names["mc_oo"]:
    analysis.draw_unscaled_histograms(on, categories=signal_category)